# TP — Explorer un labyrinthe : pile ou file ?

Vous allez écrire un programme qui trouve un chemin dans un labyrinthe, et découvrir
que le choix entre une **pile** et une **file** ne change pas seulement la vitesse du
programme : **il change ce que le programme trouve**.

Le principe de l'exploration est le suivant. On tient à jour la liste des cases
**à visiter** : celles qu'on a vues, mais dont le tour n'est pas encore venu.
À chaque tour :

1. on retire une case de la liste **à visiter**,
2. on la **visite**,
3. on ajoute ses voisins libres à la liste **à visiter**.

Le code est le même dans les deux cas. **Seule change la structure qui range les cases
à visiter** :

| Les cases à visiter sont rangées dans... | Comportement | Résultat |
|---|---|---|
| une **pile** (LIFO) | on repart toujours de la **dernière** case vue : l'exploration s'enfonce dans un couloir, puis revient sur ses pas | parcours **en profondeur** |
| une **file** (FIFO) | on repart de la **plus ancienne** : l'exploration s'étend en tache d'huile | parcours **en largeur** |

À la fin, vous aurez une animation des deux explorations et vous pourrez répondre à
la question : *laquelle trouve le plus court chemin, et à quel prix ?*

> *Au passage :* dans la littérature anglo-saxonne, cet ensemble de cases à visiter
> porte un nom, la *frontier* (ou *fringe*, ou *open list*). Vous le rencontrerez si
> vous croisez un jour les algorithmes de recherche de chemin type A\*.

---
**Plan du TP**

1. implémenter une pile et une file (par tableau de taille fixe) ;
2. écrire l'exploration, une seule fois, qui marche avec les deux ;
3. comparer, animer, mesurer.

**Ce que vous devez compléter** est signalé par `# TODO`. Chaque partie est suivie
d'une cellule de test : tant qu'elle lève une erreur, ce n'est pas bon. Le reste
(génération du labyrinthe, affichage, animation) vous est **fourni**.

---
## 0. Mise en place

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

print("numpy", np.__version__)


def leve_une_exception(f, *args):
    """Outil pour les tests : renvoie True si f(*args) leve une exception."""
    try:
        f(*args)
        return False
    except Exception:
        return True

---
## 1. La pile

Une **pile** (*stack*) range les éléments selon le principe **LIFO** : *dernier entré,
premier sorti*.

On l'implémente par un **tableau de taille fixe** `N`, comme dans le cours. L'attribut
`sommet` est l'**indice** du dernier élément entré ; il vaut `-1` quand la pile est vide.

Le constructeur vous est donné : il fixe la représentation. **À vous d'écrire les
quatre méthodes.**

Rappel du cours, dans l'ordre :

- `empiler` : on **incrémente** `sommet`, **puis** on écrit dans `tab[sommet]` ;
- `depiler` : on **décrémente** `sommet`, puis on renvoie `tab[sommet + 1]`.

Trois méthodes vous sont **fournies** et servent uniquement à l'affichage :
`contenu()`, qui renvoie le contenu sous forme de liste Python du fond vers le sommet,
`__str__()` qui permet d'écrire `print(p)`, et `tableau()` qui montre le tableau brut.
Vous n'avez pas à y toucher — mais lisez-les, elles vous seront utiles pour suivre
l'algorithme à la trace.

In [ ]:
class Pile:
    """Pile LIFO implementee par un tableau de taille fixe."""

    def __init__(self, N):
        """Cree une pile pouvant contenir au plus N elements."""
        self.N = N
        self.tab = [None] * N
        self.sommet = -1          # indice du dernier element entre (-1 = vide)

    def est_vide(self):
        """Renvoie True si la pile ne contient aucun element."""
        return self.sommet == -1

    def est_pleine(self):
        """Renvoie True si la pile occupe tout le tableau."""
        return self.sommet == self.N - 1

    def empiler(self, x):
        """Ajoute x au sommet. Leve une exception si la pile est pleine."""
        if self.est_pleine():
            raise Exception("Debordement de la pile !")
        self.sommet += 1
        self.tab[self.sommet] = x

    def depiler(self):
        """Retire et renvoie l'element au sommet.
        Leve une exception si la pile est vide."""
        if self.est_vide():
            raise Exception("La pile est vide !")
        self.sommet -= 1
        return self.tab[self.sommet + 1]

    # ------------------------------------------------------------------
    def contenu(self):
        """Contenu de la pile sous forme de liste, du FOND vers le SOMMET.
        Fourni : sert a l'affichage, la pile n'est pas modifiee."""
        return self.tab[:self.sommet + 1]

    def __str__(self):
        """Appele par print(). Fourni : ne pas modifier."""
        return f"Pile{self.contenu()}  (sommet={self.sommet})"

    def tableau(self):
        """Le tableau brut, avec un point pour les cases inutilisees. Fourni."""
        return "[" + ", ".join("." if x is None else str(x) for x in self.tab) + "]"

In [ ]:
# --- test de la pile ---
p = Pile(3)
assert p.est_vide() and not p.est_pleine()

p.empiler("a"); p.empiler("b")
assert not p.est_vide()
assert p.depiler() == "b", "une pile ressort a l'envers"

p.empiler("c"); p.empiler("d")
assert p.est_pleine(), "3 elements dans une Pile(3) : elle est pleine"
assert leve_une_exception(p.empiler, "e"), "un debordement doit lever une exception"

assert p.contenu() == ["a", "c", "d"], "contenu() va du FOND vers le SOMMET"

assert [p.depiler() for _ in range(3)] == ["d", "c", "a"]
assert p.est_vide()
assert p.contenu() == [], "une pile vide a un contenu vide"
assert leve_une_exception(p.depiler), "depiler une pile vide doit lever une exception"
print("Pile : OK")

demo = Pile(4)
demo.empiler("x"); demo.empiler("y")
print("  affichage :", demo, "| tableau :", demo.tableau())

---
## 2. La file

Une **file** (*queue*) range les éléments selon le principe **FIFO** : *premier entré,
premier sorti*.

On l'implémente par un **tableau circulaire** : quand on arrive au bout du tableau, on
revient au début. D'où deux attributs :

- `tete` : indice du prochain élément à sortir ;
- `queue` : indice du prochain emplacement libre.

**Attention au piège du cours :** une file construite avec `N` cases ne peut contenir
que **`N-1`** éléments. C'est cette case sacrifiée qui permet de distinguer « file
vide » (`tete == queue`) de « file pleine ».

Rappel, dans l'ordre :

- `enfiler` : on écrit dans `tab[queue]`, puis `queue = (queue + 1) % N` ;
- `defiler` : on renvoie `tab[tete]`, et `tete = (tete + 1) % N` ;
- pleine si `(queue + 1) % N == tete`.

Comme pour la pile, `contenu()`, `__str__()` et `tableau()` vous sont **fournies**.
Jetez un oeil à `contenu()` : elle parcourt le tableau de `tete` vers `queue` **en
bouclant**, modulo `N`. C'est le mécanisme du tampon circulaire, écrit noir sur blanc.

In [ ]:
class File:
    """File FIFO implementee par un tableau circulaire de taille fixe.

    Une File(N) contient au plus N-1 elements.
    """

    def __init__(self, N):
        """Cree une file pouvant contenir au plus N-1 elements."""
        self.N = N
        self.tab = [None] * N
        self.tete = 0             # prochain element a sortir
        self.queue = 0            # prochain emplacement libre

    def est_vide(self):
        """Renvoie True si la file ne contient aucun element."""
        return self.tete == self.queue

    def est_pleine(self):
        """Renvoie True si la file contient N-1 elements."""
        return (self.queue + 1) % self.N == self.tete

    def enfiler(self, x):
        """Ajoute x en queue. Leve une exception si la file est pleine."""
        if self.est_pleine():
            raise Exception("Debordement de la file !")
        self.tab[self.queue] = x
        self.queue = (self.queue + 1) % self.N

    def defiler(self):
        """Retire et renvoie l'element en tete.
        Leve une exception si la file est vide."""
        if self.est_vide():
            raise Exception("La file est vide !")
        x = self.tab[self.tete]
        self.tete = (self.tete + 1) % self.N
        return x

    # ------------------------------------------------------------------
    def contenu(self):
        """Contenu de la file sous forme de liste, de la TETE vers la QUEUE.
        Fourni : noter le parcours du tableau *en bouclant*, modulo N."""
        out = []
        i = self.tete
        while i != self.queue:
            out.append(self.tab[i])
            i = (i + 1) % self.N
        return out

    def __str__(self):
        """Appele par print(). Fourni : ne pas modifier."""
        return f"File{self.contenu()}  (tete={self.tete}, queue={self.queue})"

    def tableau(self):
        """Le tableau brut, avec un point pour les cases inutilisees. Fourni."""
        return "[" + ", ".join("." if x is None else str(x) for x in self.tab) + "]"

In [ ]:
# --- test de la file ---
f = File(4)                       # au plus 3 elements !
assert f.est_vide() and not f.est_pleine()

f.enfiler(1); f.enfiler(2); f.enfiler(3)
assert f.est_pleine(), "une File(4) est pleine avec 3 elements, pas 4"
assert leve_une_exception(f.enfiler, 99), "un debordement doit lever une exception"

assert f.contenu() == [1, 2, 3], "contenu() va de la TETE vers la QUEUE"

assert f.defiler() == 1, "une file ressort dans l'ordre d'arrivee"
f.enfiler(4)                      # <- ici queue repasse au debut du tableau
assert f.contenu() == [2, 3, 4], "le contenu logique ignore le bouclage du tableau"
assert [f.defiler() for _ in range(3)] == [2, 3, 4], "l'ordre FIFO doit survivre au bouclage"

assert f.est_vide()
assert f.contenu() == [], "une file vide a un contenu vide"
assert leve_une_exception(f.defiler), "defiler une file vide doit lever une exception"
print("File : OK")

### Voir la file boucler

Maintenant que `contenu()` existe, `print(f)` affiche l'état de la file. La méthode
`tableau()`, fournie, montre en plus le **tableau brut**, avec un point pour les cases
jamais utilisées.

Exécutez la cellule et lisez les trois états successifs : c'est tout le mécanisme du
tampon circulaire en six lignes.

In [ ]:
f = File(5)                     # 4 elements au maximum

for x in "abcd":
    f.enfiler(x)
print("apres avoir enfile a,b,c,d :", f)
print("                   tableau :", f.tableau())

f.defiler(); f.defiler()
print()
print("apres 2 defilages          :", f)
print("                   tableau :", f.tableau())

f.enfiler("e"); f.enfiler("f")
print()
print("apres avoir enfile e,f     :", f)
print("                   tableau :", f.tableau())

> **Question.** Dans le tableau brut de la dernière ligne, une des valeurs
> n'appartient plus à la file. Laquelle ? Pourquoi est-elle encore là ?
>
> Et où la lettre `f` a-t-elle atterri dans le tableau ? Pourquoi là ?

---
## 3. Le labyrinthe

Un labyrinthe est un simple tableau numpy d'entiers : `0` = case libre, `1` = mur.
Une case est repérée par un couple `(i, j)` : `i` la ligne, `j` la colonne.

Le générateur ci-dessous vous est fourni. Remarquez qu'il utilise lui-même **une pile**
(la variable `chemin`) : on creuse tant qu'on peut, et quand on est bloqué on dépile
pour revenir en arrière. Vous reconnaîtrez ce principe à la fin du TP.

In [ ]:
def generer_labyrinthe(k, graine=0, boucles=0.12):
    """Genere un labyrinthe de taille (2k+1) x (2k+1).

    Args:
        k (int): un labyrinthe 21x21 correspond a k = 10
        graine (int): pour obtenir toujours le meme labyrinthe
        boucles (float): proportion de murs abattus en plus. A 0, il n'existe
            qu'un seul chemin entre deux cases quelconques. Plus c'est eleve,
            plus il y a de chemins possibles.
    """
    rng = np.random.default_rng(graine)
    n = 2*k + 1
    laby = np.ones((n, n), dtype=int)

    chemin = [(1, 1)]                    # <- une pile, deja !
    laby[1, 1] = 0
    while chemin:
        i, j = chemin[-1]
        vois = [(i+di, j+dj) for di, dj in ((-2, 0), (2, 0), (0, -2), (0, 2))
                if 0 < i+di < n-1 and 0 < j+dj < n-1 and laby[i+di, j+dj] == 1]
        if not vois:
            chemin.pop()
            continue
        ni, nj = vois[rng.integers(len(vois))]
        laby[(i+ni)//2, (j+nj)//2] = 0
        laby[ni, nj] = 0
        chemin.append((ni, nj))

    # on abat quelques murs en plus : plusieurs chemins relient alors le depart
    # a l'arrivee, ce qui rendra la comparaison pile/file interessante
    murs = [(i, j) for i in range(1, n-1) for j in range(1, n-1)
            if laby[i, j] == 1 and ((laby[i-1, j] == 0 and laby[i+1, j] == 0)
                                    or (laby[i, j-1] == 0 and laby[i, j+1] == 0))]
    rng.shuffle(murs)
    for i, j in murs[:int(boucles * len(murs))]:
        laby[i, j] = 0
    return laby

### L'affichage (fourni)

`afficher()` dessine le labyrinthe. Les cases visitées sont coloriées **selon leur
ordre de visite** : violet pour les premières, jaune pour les dernières. C'est ce
dégradé qui va rendre la différence entre les deux parcours visible d'un coup d'oeil.

Vert = départ, orange = arrivée, rouge = chemin trouvé.

In [ ]:
MUR = (0.16, 0.16, 0.22)


def _image(laby, visitees, chemin=None, depart=None, arrivee=None):
    """Construit l'image RGB du labyrinthe dans son etat courant."""
    h, w = laby.shape
    img = np.ones((h, w, 3))
    img[laby == 1] = MUR
    n = max(len(visitees) - 1, 1)
    for k, (i, j) in enumerate(visitees):
        img[i, j] = plt.cm.viridis(0.15 + 0.85 * k / n)[:3]
    for (i, j) in (chemin or []):
        img[i, j] = (0.93, 0.20, 0.18)
    if depart is not None:
        img[depart] = (0.10, 0.65, 0.20)
    if arrivee is not None:
        img[arrivee] = (1.00, 0.75, 0.05)
    return img


def afficher(laby, visitees=(), chemin=None, depart=None, arrivee=None, titre="", ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 4.2))
    ax.imshow(_image(laby, list(visitees), chemin, depart, arrivee), interpolation="nearest")
    ax.set_title(titre, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    return ax


def animer(laby, visitees, chemin=None, depart=None, arrivee=None, titre="", max_images=100):
    """Renvoie un lecteur video (play / pause / vitesse) de l'exploration."""
    visitees = list(visitees)
    pas = max(1, len(visitees) // max_images)
    etapes = list(range(0, len(visitees), pas)) + [len(visitees)]
    fig, ax = plt.subplots(figsize=(4.2, 4.2))
    im = ax.imshow(_image(laby, [], None, depart, arrivee), interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(titre, fontsize=10)

    def maj(k):
        ch = chemin if k == len(etapes) - 1 else None
        im.set_data(_image(laby, visitees[:etapes[k]], ch, depart, arrivee))
        return (im,)

    anim = animation.FuncAnimation(fig, maj, frames=len(etapes), interval=90, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

In [ ]:
LABY = generer_labyrinthe(k=10, graine=1)
N_COTE = LABY.shape[0]
DEPART, ARRIVEE = (1, 1), (N_COTE - 2, N_COTE - 2)
NB_CASES_LIBRES = int((LABY == 0).sum())

print("labyrinthe :", LABY.shape, "|", NB_CASES_LIBRES, "cases libres")
afficher(LABY, depart=DEPART, arrivee=ARRIVEE, titre="le labyrinthe a explorer");

---
## 4. Se déplacer dans le labyrinthe

**À vous.** Écrivez la fonction qui donne les cases où l'on peut aller depuis une case.

On se déplace uniquement **en haut, en bas, à gauche, à droite** (pas en diagonale),
et seulement vers des cases libres (valeur `0`) situées **dans** le tableau.

In [ ]:
def voisins(laby, case):
    """Renvoie la liste des cases libres voisines de `case`.

    Args:
        laby (np.ndarray): le labyrinthe (0 = libre, 1 = mur)
        case (tuple): la case (i, j) de depart

    Returns:
        list: les couples (i, j) accessibles depuis `case`
    """
    i, j = case
    n, m = laby.shape
    out = []
    for di, dj in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        a, b = i + di, j + dj
        if 0 <= a < n and 0 <= b < m and laby[a, b] == 0:
            out.append((a, b))
    return out

In [ ]:
# --- test : petit labyrinthe fabrique a la main ---
#   . . .
#   # # .      murs en (1,0) et (1,1)
#   . . .
petit = np.array([[0, 0, 0],
                  [1, 1, 0],
                  [0, 0, 0]])

assert sorted(voisins(petit, (0, 0))) == [(0, 1)], "coin : sortir du tableau est interdit"
assert sorted(voisins(petit, (0, 2))) == [(0, 1), (1, 2)]
assert sorted(voisins(petit, (2, 0))) == [(2, 1)], "un mur n'est pas un voisin"
assert sorted(voisins(petit, (1, 2))) == [(0, 2), (2, 2)]

for c in voisins(LABY, (1, 1)):
    assert LABY[c] == 0
print("voisins : OK")

---
## 5. Où ranger les cases à visiter ?

On veut écrire **une seule** fonction d'exploration, qui marche aussi bien avec une
pile qu'avec une file. Problème : vos deux classes n'ont pas le même vocabulaire —
`empiler`/`depiler` d'un côté, `enfiler`/`defiler` de l'autre.

On les enveloppe donc dans deux petites classes qui parlent, elles, le **même**
langage : `ajouter`, `retirer`, `est_vide`. L'exploration dira simplement « ajoute
cette case à visiter » sans savoir ce qu'il y a derrière — et c'est ce qui permettra
de changer de stratégie en changeant **un seul mot**.

**À vous.** Complétez les deux classes. Chacune tient en trois lignes.

In [ ]:
class PileAVisiter:
    """Range les cases a visiter dans une PILE : la derniere arrivee ressort en premier."""

    def __init__(self, N):
        self.s = Pile(N)

    def ajouter(self, case):
        self.s.empiler(case)

    def retirer(self):
        return self.s.depiler()

    def est_vide(self):
        return self.s.est_vide()


class FileAVisiter:
    """Range les cases a visiter dans une FILE : la premiere arrivee ressort en premier."""

    def __init__(self, N):
        self.s = File(N)

    def ajouter(self, case):
        self.s.enfiler(case)

    def retirer(self):
        return self.s.defiler()

    def est_vide(self):
        return self.s.est_vide()

In [ ]:
# --- test : meme vocabulaire, comportements opposes ---
p = PileAVisiter(10)
for x in ["a", "b", "c"]:
    p.ajouter(x)
assert [p.retirer() for _ in range(3)] == ["c", "b", "a"], "une pile ressort a l'envers"

f = FileAVisiter(10)
for x in ["a", "b", "c"]:
    f.ajouter(x)
assert [f.retirer() for _ in range(3)] == ["a", "b", "c"], "une file ressort dans l'ordre"
print("PileAVisiter et FileAVisiter : OK")

> **Question 1.** Une case n'entre qu'une seule fois dans `a_visiter` (on le
> garantira dans l'exploration). Quelle taille `N` faut-il donc donner à `PileAVisiter` pour ne jamais
> déborder sur ce labyrinthe ? Et à `FileAVisiter` — attention, elle ne contient que
> `N-1` éléments.

---
## 6. L'exploration

Voici le coeur du TP. Avant le pseudo-code, fixons le **vocabulaire** : ces mots sont
les mêmes dans le texte et dans le code.

| Dans le texte | Dans le code | Ce que c'est |
|---|---|---|
| les cases **à visiter** | `a_visiter` | la pile ou la file : les cases découvertes dont le tour n'est pas encore venu |
| les cases **découvertes** | `parents` | pour chaque case découverte, on y note d'où l'on y est arrivé |
| les cases **visitées** | `visitees` | la liste des cases déjà ressorties et traitées, dans l'ordre : c'est le résultat |

Le pseudo-code :

```
parents[depart] = rien                       # le depart est decouvert
mettre le depart dans a_visiter

tant que a_visiter n'est pas vide :

    case = retirer une case de a_visiter     # la pile ou la file choisit laquelle
    ajouter case a la liste visitees
    si case est l'arrivee : on a fini

    pour chaque voisin libre de case :
        si le voisin n'est pas dans parents :   # donc pas encore decouvert
            parents[voisin] = case              # on note d'ou on arrive
            mettre le voisin dans a_visiter
```

**Découverte et visite, ce n'est pas la même chose.** Une case est *découverte* quand
on la met dans `a_visiter` ; elle est *visitée* quand on l'en ressort. Entre les deux,
elle attend son tour — et c'est exactement à cela que sert `a_visiter`. Dans la trace
de la section suivante, vous verrez des cases rester plusieurs tours dans la liste
avant d'être visitées.

Le test avant d'ajouter un voisin doit donc porter sur la **découverte**, pas sur la
visite : sinon une case déjà présente dans `a_visiter` y serait ajoutée une deuxième
fois par un autre de ses voisins, et donc traitée deux fois.

**Et cela ne coûte aucune structure supplémentaire :** `parents` contient exactement
les cases découvertes, puisqu'on y écrit au moment même où on les découvre. Le test
« le voisin est-il déjà dans `parents` ? » fait d'une pierre deux coups.

**À vous.**

In [ ]:
def explorer(laby, depart, arrivee, a_visiter):
    """Explore le labyrinthe depuis `depart`.

    Args:
        a_visiter: un objet PileAVisiter ou FileAVisiter, deja construit.
            C'est lui, et lui seul, qui decide de la strategie d'exploration.

    Returns:
        (visitees, parents) : la liste des cases dans leur ordre de visite,
        et le dictionnaire {case decouverte: case d'ou l'on vient}
    """
    parents = {depart: None}       # le depart est decouvert, il ne vient de nulle part
    visitees = []

    a_visiter.ajouter(depart)
    while not a_visiter.est_vide():
        case = a_visiter.retirer()
        visitees.append(case)
        if case == arrivee:
            break
        for v in voisins(laby, case):
            if v not in parents:        # le voisin n'est pas encore decouvert
                parents[v] = case
                a_visiter.ajouter(v)

    return visitees, parents

In [ ]:
# --- test ---
visitees_p, parents_p = explorer(LABY, DEPART, ARRIVEE, PileAVisiter(NB_CASES_LIBRES))
visitees_f, parents_f = explorer(LABY, DEPART, ARRIVEE, FileAVisiter(NB_CASES_LIBRES + 1))

for nom, visitees in [("pile", visitees_p), ("file", visitees_f)]:
    assert visitees[0] == DEPART, "on commence par le depart"
    assert visitees[-1] == ARRIVEE, "on doit s'arreter sur l'arrivee"
    assert len(set(visitees)) == len(visitees), "une case ne doit etre visitee qu'une fois"
    print(f"{nom} : {len(visitees)} cases visitees")

### Voir l'algorithme travailler

On aimerait suivre ce qui se passe **pendant** l'exploration. Plutôt que de modifier
`explorer`, on réutilise l'idée de la section 5 : on **enveloppe** le rangement dans
un objet qui parle le même langage (`ajouter`, `retirer`, `est_vide`) mais qui raconte
tout ce qu'on lui fait.

`explorer` ne verra pas la différence.

In [ ]:
class Bavard:
    """Enveloppe un PileAVisiter ou un FileAVisiter et raconte ce qui s'y passe.

    Fourni : a utiliser tel quel.
    """

    def __init__(self, rangement, max_lignes=60):
        self.r = rangement
        self.etape = 0
        self.lignes = 0
        self.max_lignes = max_lignes

    def _dire(self, texte):
        if self.lignes < self.max_lignes:
            print(texte)
        elif self.lignes == self.max_lignes:
            print("    ... (la suite est masquee)")
        self.lignes += 1

    def ajouter(self, case):
        self.r.ajouter(case)
        self._dire(f"         + {case}   ->   {self.r.s}")

    def retirer(self):
        case = self.r.retirer()
        self.etape += 1
        self._dire(f"{self.etape:3d}. on visite {case}")
        return case

    def est_vide(self):
        return self.r.est_vide()

Sur le grand labyrinthe l'affichage serait illisible : on prend un tout petit
labyrinthe.

In [ ]:
PETIT = generer_labyrinthe(k=3, graine=1, boucles=0.15)
P_DEPART = (1, 1)
P_ARRIVEE = (PETIT.shape[0] - 2, PETIT.shape[0] - 2)
P_LIBRES = int((PETIT == 0).sum())

print("petit labyrinthe :", PETIT.shape, "|", P_LIBRES, "cases libres")
afficher(PETIT, depart=P_DEPART, arrivee=P_ARRIVEE, titre="petit labyrinthe");

In [ ]:
print("========== a_visiter = PILE ==========")
_ = explorer(PETIT, P_DEPART, P_ARRIVEE, Bavard(PileAVisiter(P_LIBRES)))

In [ ]:
print("========== a_visiter = FILE ==========")
_ = explorer(PETIT, P_DEPART, P_ARRIVEE, Bavard(FileAVisiter(P_LIBRES + 1)))

> **Question.** Comparez les deux traces. Dans un cas, la case qu'on visite est
> celle qui vient tout juste d'être ajoutée ; dans l'autre, c'est une case ajoutée il y
> a longtemps. Repérez-le sur les listes affichées.
>
> Regardez aussi la **longueur** de la liste au fil du temps : laquelle des deux
> structures gonfle le plus ? Qu'est-ce que cela dit de la mémoire consommée par chaque
> parcours ?

---
## 7. Reconstituer le chemin

`explorer` ne renvoie pas le chemin : elle renvoie `parents`, qui dit pour chaque case
d'où l'on venait. Pour retrouver le chemin, on part de l'**arrivée** et on remonte de
parent en parent jusqu'au **départ** — puis on retourne la liste.

**À vous.**

In [ ]:
def reconstruire_chemin(parents, depart, arrivee):
    """Remonte les parents depuis l'arrivee jusqu'au depart.

    Returns:
        list: les cases du chemin, du depart vers l'arrivee.
              Liste vide si l'arrivee n'a pas ete atteinte.
    """
    if arrivee not in parents:          # l'arrivee n'a jamais ete decouverte
        return []

    chemin, case = [arrivee], arrivee
    while case != depart:
        case = parents[case]
        chemin.append(case)
    chemin.reverse()
    return chemin

In [ ]:
# --- test ---
chemin_p = reconstruire_chemin(parents_p, DEPART, ARRIVEE)
chemin_f = reconstruire_chemin(parents_f, DEPART, ARRIVEE)

for nom, ch in [("pile", chemin_p), ("file", chemin_f)]:
    assert ch[0] == DEPART and ch[-1] == ARRIVEE
    for a, b in zip(ch, ch[1:]):
        assert abs(a[0]-b[0]) + abs(a[1]-b[1]) == 1, "deux cases consecutives sont voisines"
        assert LABY[b] == 0, "le chemin ne traverse pas les murs"
    print(f"{nom} : chemin valide de {len(ch)} cases")

---
## 8. Le résultat : pile contre file

Tout est prêt. Comparons les deux explorations côte à côte.

Rappel du code couleur : le dégradé **violet → jaune** donne l'ordre de visite,
le **rouge** est le chemin trouvé, le vert le départ, l'orange l'arrivée.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))

afficher(LABY, visitees_p, chemin_p, DEPART, ARRIVEE, ax=axes[0],
         titre=f"PILE  (profondeur)\n{len(visitees_p)} cases explorees, chemin de {len(chemin_p)}")
afficher(LABY, visitees_f, chemin_f, DEPART, ARRIVEE, ax=axes[1],
         titre=f"FILE  (largeur)\n{len(visitees_f)} cases explorees, chemin de {len(chemin_f)}")

plt.tight_layout()

### L'exploration en mouvement

Les deux cellules suivantes produisent un **lecteur vidéo** : boutons lecture / pause,
image par image, réglage de vitesse. Regardez surtout *la forme* de l'exploration.

In [ ]:
animer(LABY, visitees_p, chemin_p, DEPART, ARRIVEE, titre="a_visiter = PILE  (parcours en profondeur)")

In [ ]:
animer(LABY, visitees_f, chemin_f, DEPART, ARRIVEE, titre="a_visiter = FILE  (parcours en largeur)")

> **Question 2.** Décrivez en une phrase la forme de chaque exploration. Laquelle
> avance « en serpentant », laquelle « en vagues concentriques » ? Pourquoi ?
>
> **Question 3.** Complétez ce tableau avec vos résultats :
>
> | | cases explorées | longueur du chemin |
> |---|---|---|
> | pile (profondeur) | ... | ... |
> | file (largeur) | ... | ... |
>
> **Question 4.** L'une explore **moins** de cases, l'autre trouve un chemin **plus
> court**. Laquelle fait quoi ? Est-ce un hasard de ce labyrinthe, ou est-ce toujours
> vrai ? Relancez avec `graine=2`, `graine=3`... pour vous faire une idée.
>
> **Question 5.** Un GPS qui calcule un itinéraire : lequel des deux parcours
> utiliseriez-vous ? Et un solveur qui doit seulement dire *s'il existe* une sortie,
> avec le moins de mémoire possible ?

In [ ]:
# Espace libre : relancez l'experience avec d'autres labyrinthes
# (changez la graine, la taille k, ou le taux de boucles)

# TODO (facultatif) : votre experimentation ici

---
## 9. La taille fixe, en vrai

Votre pile est un **tableau de taille fixe**. Que se passe-t-il si on la dimensionne
trop petit ? Essayons avec 20 cases seulement.

In [ ]:
try:
    explorer(LABY, DEPART, ARRIVEE, PileAVisiter(20))
except Exception as e:
    print("Exception levee :", type(e).__name__, "-", e)

> **Question 6.** Ce n'est pas un bug, c'est la limite annoncée par le cours.
> Trois façons de s'en sortir :
>
> 1. dimensionner le tableau au pire cas — combien vaut-il ici ?
> 2. utiliser une pile implémentée par une **liste chaînée**, de taille arbitraire ;
> 3. agrandir le tableau quand il est plein, en recopiant tout — c'est ce que fait
>    une liste Python, et c'est pourquoi `append` est en $O(1)$ *amorti*.
>
> **Question 7.** La solution 2 supprime-t-elle vraiment *toute* limite de taille ?

### Pour aller plus loin (facultatif)

Écrivez une pile par **liste chaînée** — une cellule = une valeur + un pointeur vers
la suivante, et `sommet` désigne la première cellule. Elle n'a pas besoin de connaître
`N` à la construction. Branchez-la dans une troisième classe de rangement et vérifiez
que l'exploration donne **exactement le même résultat**.

In [ ]:
# TODO (facultatif)
#
# class Cellule:
#     def __init__(self, valeur, suivante=None): ...
#
# class PileChainee:
#     def __init__(self, N=None): self.sommet = None
#     def est_vide(self): ...
#     def empiler(self, x): ...
#     def depiler(self): ...
#
# class PileChaineeAVisiter:
#     ...
#
# visitees_lc, _ = explorer(LABY, DEPART, ARRIVEE, PileChaineeAVisiter())
# assert visitees_lc == visitees_p

---
## 10. Pourquoi pas une simple liste Python ?

Objection légitime : « tout ça pour ça, une liste Python aurait suffi ». Voyons le prix.

La classe ci-dessous se comporte **exactement** comme une pile, mais elle ajoute et
retire ses éléments **au début** de la liste, avec `insert(0, x)` et `pop(0)`. Or
insérer au début d'un tableau oblige à décaler tout le reste : c'est la ligne
`ajouter/insérer → O(n)` du tableau de complexité de votre cours.

In [ ]:
class PileAVisiterLente:
    """Meme comportement qu'une PileAVisiter, mais chaque operation est en O(n)."""

    def __init__(self, N=None):
        self.l = []

    def ajouter(self, case):
        self.l.insert(0, case)     # decale tout le reste : O(n)

    def retirer(self):
        return self.l.pop(0)       # decale tout le reste : O(n)

    def est_vide(self):
        return len(self.l) == 0

In [ ]:
# elle donne bien le meme resultat que votre pile : c'est un programme *juste*
visitees_lent, _ = explorer(LABY, DEPART, ARRIVEE, PileAVisiterLente())
assert visitees_lent == visitees_p
print("PileAVisiterLente explore exactement comme PileAVisiter :", len(visitees_lent), "cases")

La cellule suivante, **fournie**, mesure le coût de `n` ajouts suivis de `n` retraits
pour les deux structures, et pour des `n` croissants. Exécutez-la et observez les
nombres avant de passer au graphique.

In [ ]:
def mesurer(Rangement, n):
    """Chronometre n ajouts puis n retraits. Renvoie la duree en secondes.

    Fourni.
    """
    r = Rangement(n + 1)
    t0 = time.perf_counter()
    for k in range(n):
        r.ajouter((k, k))
    while not r.est_vide():
        r.retirer()
    return time.perf_counter() - t0


TAILLES = [1000, 2000, 4000, 8000, 16000, 32000]

temps_pile, temps_lente = [], []
for n in TAILLES:
    temps_pile.append(mesurer(PileAVisiter, n))
    temps_lente.append(mesurer(PileAVisiterLente, n))
    print(f"n = {n:6d} :  PileAVisiter = {temps_pile[-1]*1000:8.2f} ms"
          f"   |  PileAVisiterLente = {temps_lente[-1]*1000:8.2f} ms")

In [ ]:
plt.figure(figsize=(5.5, 4))
plt.plot(TAILLES, temps_pile, "o-", label="pile (tableau) : $O(1)$ par operation")
plt.plot(TAILLES, temps_lente, "s-", label="liste, insert(0)/pop(0) : $O(n)$ par operation")
plt.xlabel("nombre d'elements $n$")
plt.ylabel("temps total (s)")
plt.title("Le prix d'une mauvaise structure")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()

> **Question 8.** Quelle est l'allure de chaque courbe ? Si chaque opération coûte
> $O(1)$, le total pour $n$ opérations est en $O(n)$ : une droite. Et si chaque
> opération coûte $O(n)$ ?
>
> **Question 9.** Quand `n` double, par combien est multiplié chacun des deux temps ?
> Vérifiez sur vos mesures.
>
> **Question 10.** `PileAVisiterLente` donne exactement le même résultat d'exploration
> que `PileAVisiter` — on l'a vérifié par un `assert`. Un programme *juste* peut donc
> être inutilisable. Reformulez en une phrase ce que « développement efficace » veut dire.

---
## Ce qu'il faut retenir

- La structure qui range les **cases à visiter** décide de la stratégie : une pile
  explore **en profondeur**, une file explore **en largeur**. Le reste du code est
  identique — c'est le sens des classes `PileAVisiter` et `FileAVisiter`.
- Le parcours en largeur trouve le **plus court chemin**, mais explore davantage.
  Le parcours en profondeur trouve *un* chemin en visitant moins de cases.
- Un tableau de taille fixe **déborde** : il faut soit majorer le besoin, soit passer
  à une structure de taille arbitraire, soit réallouer.
- Deux programmes qui donnent le même résultat peuvent avoir des coûts sans commune
  mesure. C'est le choix de la structure de données qui fait la différence.